## Task 1 - Data Analysis and Preparation

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# 1. Load Dataset
df = pd.read_csv("AB_NYC_2019.csv")
print("Original Dataset Shape:", df.shape)

# 2. Data Cleaning & Outlier Removal (Filter extreme price outliers)
df_clean = df[(df["price"] >= 10) & (df["price"] <= 500)].copy()
df_clean = df_clean[df_clean["minimum_nights"] <= 30]
print("Shape after removing price and night outliers:", df_clean.shape)

# 3. Feature Selection & Target Transformation
# Drop non-predictive metadata (IDs, names, text descriptions)
X = df_clean.drop(columns=["id", "name", "host_id", "host_name", "last_review", "price"])

# Log-transform target variable to fix right-skewness
y = np.log1p(df_clean["price"])

# 4. Define Numeric and Categorical Columns
num_cols = [
    "latitude", 
    "longitude", 
    "minimum_nights", 
    "number_of_reviews", 
    "reviews_per_month", 
    "calculated_host_listings_count", 
    "availability_365"
]
cat_cols = ["neighbourhood_group", "room_type"]

# 5. Build Preprocessing Pipelines
num_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

cat_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

preprocessor = ColumnTransformer(transformers=[
    ("num", num_transformer, num_cols),
    ("cat", cat_transformer, cat_cols)
])

# 6. Train-Test Split (80% Train, 20% Test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Fit & transform features
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print("Task 1 complete!")
print("Processed Training Feature Matrix Shape:", X_train_processed.shape)

Original Dataset Shape: (48895, 16)
Shape after removing price and night outliers: (47125, 16)
Task 1 complete!
Processed Training Feature Matrix Shape: (37700, 15)


## Task 2 - Model Training and Evaluation

In [3]:
import os
import joblib
import numpy as np
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# 1. Define Candidate Models
# Added max_depth=15 to RandomForestRegressor to prevent uncompressed tree growth
models = {
    "Linear Regression": LinearRegression(),
    "Decision Tree": DecisionTreeRegressor(random_state=42, max_depth=10),
    "Random Forest": RandomForestRegressor(n_estimators=100, max_depth=15, random_state=42, n_jobs=-1)
}

results = []

# 2. Train and Evaluate Each Model Pipeline
for name, regressor in models.items():
    # Build complete end-to-end pipeline combining preprocessor + regressor
    full_pipeline = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("regressor", regressor)
    ])
    
    # Train pipeline on raw X_train
    full_pipeline.fit(X_train, y_train)
    
    # Predict on raw X_test
    y_pred_log = full_pipeline.predict(X_test)
    
    # Inverse log transformation (convert back to actual dollar amounts)
    y_pred = np.expm1(y_pred_log)
    y_true = np.expm1(y_test)
    
    # Calculate regression metrics
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    
    results.append({
        "Model": name,
        "MAE ($)": round(mae, 2),
        "RMSE ($)": round(rmse, 2),
        "R2 Score": round(r2, 4)
    })

# Convert performance metric summary to DataFrame
results_df = pd.DataFrame(results)
print("--- Model Comparison ---")
print(results_df.to_string(index=False))

# 3. Select and Save Best Performing Model Pipeline (Random Forest)
best_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("regressor", RandomForestRegressor(n_estimators=100, max_depth=15, random_state=42, n_jobs=-1))
])
best_pipeline.fit(X_train, y_train)

# Ensure models directory exists
os.makedirs("models", exist_ok=True)
pipeline_path = "models/airbnb_pipeline.pkl"

# Save compressed pipeline to disk (compress=3 drops file size from >100MB down to ~15-20MB)
joblib.dump(best_pipeline, pipeline_path, compress=3)

file_size_mb = os.path.getsize(pipeline_path) / (1024 * 1024)
print(f"\nTask 2 complete! Pipeline saved to '{pipeline_path}'")
print(f"File Size: {file_size_mb:.2f} MB (Successfully compressed for GitHub push)")

--- Model Comparison ---
            Model  MAE ($)  RMSE ($)  R2 Score
Linear Regression    43.94     69.35    0.3940
    Decision Tree    41.27     65.77    0.4549
    Random Forest    38.76     62.47    0.5084

Task 2 complete! Pipeline saved to 'models/airbnb_pipeline.pkl'
File Size: 21.73 MB (Successfully compressed for GitHub push)
